# **Экспорт моделей для ios**

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import torchvision.transforms as transforms
from PIL import Image
# %pip install coremltools
import coremltools as ct

## **Сегментация**

`ResNetFeatMask` — **тот же forward, что в `etalon.py`**: вход маски `[1, 1, 256, 256]`, внутри `F.interpolate` nearest до 8×8 на карте признаков.

In [ ]:
class ResNetFeatMask(nn.Module):
    def __init__(self, num_classes=2, pretrained_backbone=True):
        super().__init__()
        backbone = models.resnet50(pretrained=pretrained_backbone)
        self.features = nn.Sequential(*(list(backbone.children())[:-2]))
        self.avgpool = backbone.avgpool
        self.fc = nn.Linear(backbone.fc.in_features, num_classes)

    def forward(self, x, seg_mask):
        feat = self.features(x)
        mask_small = F.interpolate(seg_mask, size=feat.shape[2:], mode="nearest")
        feat = feat * mask_small
        out = self.avgpool(feat)
        out = torch.flatten(out, 1)
        return self.fc(out)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Загрузка модели

In [12]:
resnet_file = "best_f1_resnet_3.05.25.pth"

class_model = ResNetFeatMask(num_classes=2, pretrained_backbone=True).to(device)
class_model.load_state_dict(torch.load(resnet_file, map_location=device))
class_model.to(device)

ResNetFeatMask(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Con

Трансформации

In [7]:
class_image_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
# Как `etalon.__class_mask_transform`: 256×256 + бинаризация; даунскейл до 8×8 делает сама модель.
class_mask_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
    transforms.Lambda(lambda t: (t > 0.5).float()),
])

In [ ]:

fake_image = Image.new("RGB", (256, 256), color="gray")
fake_mask = Image.new("L", (256, 256), color=0)

img_tensor = class_image_transform(fake_image).unsqueeze(0)
mask_tensor = class_mask_transform(fake_mask).unsqueeze(0)

model_cpu = class_model.cpu().eval()
img_tensor = img_tensor.cpu()
mask_tensor = mask_tensor.cpu()

assert mask_tensor.shape == (1, 1, 256, 256), (
    f"Ожидается маска [1,1,256,256] как в etalon.__class_mask_transform; got {tuple(mask_tensor.shape)}. "
    "Старый экспорт [1,2048,8,8] несовместим с текущим iOS."
)

print(model_cpu(img_tensor, mask_tensor).shape)
print(img_tensor.dtype, mask_tensor.dtype)
print(img_tensor.shape, mask_tensor.shape)

traced_model = torch.jit.trace(model_cpu, (img_tensor, mask_tensor))

mlmodel = ct.convert(
    traced_model,
    inputs=[
        ct.TensorType(name="image", shape=img_tensor.shape),
        ct.TensorType(name="mask", shape=mask_tensor.shape),
    ],
    minimum_deployment_target=ct.target.iOS15,
)
mlmodel.save("classification_model.mlpackage")

torch.Size([1, 2])
torch.float32
torch.float32
torch.Size([1, 3, 256, 256])
torch.Size([1, 2048, 8, 8])


Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 125.04 passes/s]


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
%cd /content/classification_model.mlpackage

/content/classification_model.mlpackage


In [18]:
!zip -r models.zip /content/classification_model.mlpackage

  adding: content/classification_model.mlpackage/ (stored 0%)
  adding: content/classification_model.mlpackage/Manifest.json (deflated 60%)
  adding: content/classification_model.mlpackage/Data/ (stored 0%)
  adding: content/classification_model.mlpackage/Data/com.apple.CoreML/ (stored 0%)
  adding: content/classification_model.mlpackage/Data/com.apple.CoreML/model.mlmodel (deflated 89%)
  adding: content/classification_model.mlpackage/Data/com.apple.CoreML/weights/ (stored 0%)
  adding: content/classification_model.mlpackage/Data/com.apple.CoreML/weights/weight.bin (deflated 8%)


In [19]:
from google.colab import files
files.download('models.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>